# 🚕 NYC HVFHV Taxi Demand-Supply Imbalance Prediction

**Project**: Predict demand-supply imbalance for driver reallocation  
**Data**: NYC High Volume FHV (HVFHV) trip records 2023–2025 (Parquet)  
**Aggregation**: 30-minute sliding window, sliding every 5 minutes  
**Grain**: `(zone_id, window_end_time)`

---
## Section 1 — Setup

In [1]:
# ─── Standard Library ────────────────────────────────────────────────────────
import os
import warnings
warnings.filterwarnings('ignore')

# ─── PySpark ─────────────────────────────────────────────────────────────────
from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.functions import count, when, col


# ─── Data / ML ───────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import lightgbm as lgb

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (
    classification_report, roc_auc_score,
    confusion_matrix, ConfusionMatrixDisplay,
    mean_squared_error, r2_score
)
from sklearn.preprocessing import label_binarize

# ─── Visualisation ────────────────────────────────────────────────────────────
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

matplotlib.rcParams.update({
    'figure.dpi': 120,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
})
PALETTE = sns.color_palette('tab10')

print('All imports OK')

All imports OK


In [2]:
# ─── Spark Session ────────────────────────────────────────────────────────────
spark = (
    SparkSession.builder
    .appName('NYC_HVFHV_Demand_Supply')

    # Core
    .master("local[*]")

    # Memory (quan trọng)
    .config("spark.driver.memory", "8g")
    .config("spark.driver.maxResultSize", "4g")

    # Shuffle tuning (RẤT QUAN TRỌNG)
    .config("spark.sql.shuffle.partitions", "64")

    # Adaptive Query Execution
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")

    # Parquet handling
    .config("spark.sql.parquet.mergeSchema", "false")
    .config("spark.sql.parquet.filterPushdown", "true")

    # Timezone
    .config("spark.sql.session.timeZone", "America/New_York")

    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print(f'Spark version: {spark.version}')

Spark version: 3.5.7


In [3]:
BRONZE_PATH = 'datasets/fvhfv/2025/*'
schema = StructType([
    StructField("hvfhs_license_num", StringType(), True),
    StructField("dispatching_base_num", StringType(), True),
    StructField("originating_base_num", StringType(), True),
    StructField("PULocationID", IntegerType(), True),
    StructField("DOLocationID", IntegerType(), True),
    StructField("request_datetime", TimestampNTZType(), True),
    StructField("pickup_datetime", TimestampNTZType(), True),
    StructField("dropoff_datetime", TimestampNTZType(), True),
    StructField("trip_miles", DoubleType(), True),
    StructField("trip_time", LongType(), True),
    StructField("base_passenger_fare", DoubleType(), True),
    StructField("driver_pay", DoubleType(), True),
    StructField("tips", DoubleType(), True),
    StructField("congestion_surcharge", DoubleType(), True),
    StructField("airport_fee", DoubleType(), True),
    StructField("tolls", DoubleType(), True),
    StructField("bcf", DoubleType(), True),
    StructField("sales_tax", DoubleType(), True),
    StructField("shared_match_flag", StringType(), True),
    StructField("shared_request_flag", StringType(), True),
    StructField("on_scene_datetime", TimestampNTZType(), True),
    StructField("access_a_ride_flag", StringType(), True),
    StructField("wav_request_flag", StringType(), True),
    StructField("wav_match_flag", StringType(), True)
])
df = (
    spark.read
    # .schema(schema)
    .parquet(BRONZE_PATH)
)
df.count(), len(df.columns)

(243589684, 25)

In [4]:
trip = df.select(
    col("PULocationID").cast("long"),
    col("DOLocationID").cast("long"),
    col("request_datetime"),
    col("pickup_datetime"),
    col("dropoff_datetime"),
    col("trip_miles").cast("double"),
    col("trip_time").cast("long"),
    col("base_passenger_fare").cast("double"),
    col("driver_pay").cast("double"),
    col("tips").cast("double"),
    col("congestion_surcharge").cast("double"),
    col("airport_fee").cast("double"),
    col("tolls").cast("double"),
    col("bcf").cast("double"),
    col("sales_tax").cast("double"),
    col("shared_match_flag").alias("share_match_flag"),
    col("shared_request_flag").alias("share_request_flag"),
    col("wav_request_flag"),
    col("access_a_ride_flag"),
    col("wav_match_flag"),
    col("hvfhs_license_num"),
)

In [5]:
null_stats = trip.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in trip.columns
])

# null_stats.show()

In [6]:
# trip.describe().show()

In [7]:
trip.groupby("wav_match_flag") \
    .count().show()

+--------------+---------+
|wav_match_flag|    count|
+--------------+---------+
|             N|222207859|
|             Y| 21381825|
+--------------+---------+



In [8]:
trip.where("tips ==0").show(10, truncate=False)
trip.where("tips ==0").count()

+------------+------------+-------------------+-------------------+-------------------+----------+---------+-------------------+----------+----+--------------------+-----------+-----+----+---------+----------------+------------------+----------------+------------------+--------------+-----------------+
|PULocationID|DOLocationID|request_datetime   |pickup_datetime    |dropoff_datetime   |trip_miles|trip_time|base_passenger_fare|driver_pay|tips|congestion_surcharge|airport_fee|tolls|bcf |sales_tax|share_match_flag|share_request_flag|wav_request_flag|access_a_ride_flag|wav_match_flag|hvfhs_license_num|
+------------+------------+-------------------+-------------------+-------------------+----------+---------+-------------------+----------+----+--------------------+-----------+-----+----+---------+----------------+------------------+----------------+------------------+--------------+-----------------+
|148         |211         |2025-01-01 00:28:07|2025-01-01 00:33:25|2025-01-01 00:54:24|1

197468886

In [9]:
schemas = {}

for f in trip.inputFiles():
    df = spark.read.parquet(f)
    schemas[f] = df.schema.simpleString()

# group theo schema
from collections import defaultdict

groups = defaultdict(list)

for f, s in schemas.items():
    groups[s].append(f)

print(f"Total different schemas: {len(groups)}")

for i, (schema, files) in enumerate(groups.items()):
    print(f"\n--- Schema group {i+1} ({len(files)} files) ---")
    print(schema)

Total different schemas: 1

--- Schema group 1 (12 files) ---
struct<hvfhs_license_num:string,dispatching_base_num:string,originating_base_num:string,request_datetime:timestamp_ntz,on_scene_datetime:timestamp_ntz,pickup_datetime:timestamp_ntz,dropoff_datetime:timestamp_ntz,PULocationID:int,DOLocationID:int,trip_miles:double,trip_time:bigint,base_passenger_fare:double,tolls:double,bcf:double,sales_tax:double,congestion_surcharge:double,airport_fee:double,tips:double,driver_pay:double,shared_request_flag:string,shared_match_flag:string,access_a_ride_flag:string,wav_request_flag:string,wav_match_flag:string,cbd_congestion_fee:double>


## Section 1.5 -- Clean data

In [10]:
trips = trip.where("base_passenger_fare > 0 and driver_pay > 0 and trip_miles < 500 and trip_time < 20000 and driver_pay < 1500 ")
trips.count()

243149177

---
## Section 2 — Feature Engineering (Spark)

In [12]:
# ─── Window Configuration ─────────────────────────────────────────────────────
WINDOW_SECONDS = 30 * 60   # 30 min in seconds
SLIDE_SECONDS  =  5 * 60   # 5 min  in seconds

def epoch_to_window_end(ts_col: str, window_s: int = WINDOW_SECONDS, slide_s: int = SLIDE_SECONDS):
    """
    Assign each timestamp to the next (future) window boundary.
    window_end = ceil(ts / slide) * slide  — snaps to the next slide boundary.
    Spark's window() function handles this natively.
    """
    return F.window(F.col(ts_col), f'{window_s} seconds', f'{slide_s} seconds')

flag = lambda c: F.when(F.col(c) == "Y", 1).otherwise(0)
is_uber = F.when(F.col("hvfhs_license_num") == "HV0003", 1).otherwise(0)
is_lyft = F.when(F.col("hvfhs_license_num") == "HV0005", 1).otherwise(0)

print('Window helpers defined')

Window helpers defined


In [13]:
demand = (
    trips
    .withColumn("zone_id", F.col("PULocationID"))
    .withColumn("window", epoch_to_window_end("request_datetime"))
    .groupBy("zone_id", "window")
    .agg(
        # volume
        F.count("*").alias("requests_30m"),
        F.sum(
            F.when(
                F.col("request_datetime") >= F.col("window.end") - F.expr(f"INTERVAL {SLIDE_SECONDS} seconds"),
                1
            ).otherwise(0)
        ).alias("requests_5m"),

        # WAV / AAR (request-time)
        F.sum(flag("wav_request_flag")).alias("wav_req_30m"),
        F.sum(flag("access_a_ride_flag")).alias("aar_req_30m"),
        F.sum(flag("share_request_flag")).alias("share_req_30m"),

        # platform share
        F.sum(is_uber).alias("uber_req"),
        F.sum(is_lyft).alias("lyft_req"),
    )
    .withColumn("window_end", F.col("window.end"))
    .drop("window")
).where("zone_id <= 263 and window_end <= '2025-12-31 23:30:00' and window_end >= '2025-01-01 00:30:00'")


In [14]:
pickup = (
    trips
    .withColumn("zone_id", F.col("PULocationID"))
    .withColumn("window", epoch_to_window_end("pickup_datetime"))
    .withColumn(
        "pickup_delay_s",
        F.unix_timestamp("pickup_datetime") - F.unix_timestamp("request_datetime")
    )
    .groupBy("zone_id", "window")
    .agg(
        # volume
        F.count("*").alias("pickup_30m"),
        F.sum(
            F.when(
                F.col("pickup_datetime") >= F.col("window.end") - F.expr(f"INTERVAL {SLIDE_SECONDS} seconds"),
                1
            )
        ).alias("pickup_5m"),

        # delay
        F.mean("pickup_delay_s").alias("pickup_delay_mean"),
        F.stddev("pickup_delay_s").alias("pickup_delay_std"),
        
        # matching (proxy)
        F.sum(
            F.when(col('request_datetime') >= col('window.start') , 1).otherwise(0)
        ).alias('matched_rp'),
        
        F.sum(
            F.when(((col('request_datetime') >= col('window.start') )
                   & (F.col("pickup_datetime") >= (F.col("window.end") - F.expr(f"INTERVAL {SLIDE_SECONDS} seconds")))), 1).otherwise(0)
        ).alias('matched_rp_5m'),

        # WAV match
        F.sum(flag("wav_match_flag")).alias("wav_match_30m"),
        F.sum(flag("share_match_flag")).alias("share_match_30m"),

    )
    .withColumn("window_end", F.col("window.end"))
    .drop("window")
).where("zone_id <= 263 and window_end <= '2025-12-31 23:30:00' and window_end >= '2025-01-01 00:30:00'")
# pickup.printSchema(), pickup.show(5, truncate=False), pickup.count()
# pickup.select(F.min("window_end"), F.max("window_end")).show()

In [ ]:
dropoff_counts = (
    trips
    .withColumn("zone_id", F.col("DOLocationID"))
    .withColumn("window", epoch_to_window_end("dropoff_datetime"))
    .groupBy("zone_id", "window")
    .agg(
        F.count("*").alias("dropoff_30m"),
        F.sum(
            F.when(
                F.col("dropoff_datetime") >= F.col("window.end") - F.expr(f"INTERVAL {SLIDE_SECONDS} seconds"),
                1
            )
        ).alias("dropoff_5m"),
        
        F.sum(
            F.when(((col('request_datetime') >= col('window.start'))), 1).otherwise(0)
        ).alias('matched_rd'),
        
        F.sum(
            F.when(((col('pickup_datetime') >= col('window.start'))), 1).otherwise(0)
        ).alias('matched_pd'),
    )
    .withColumn("window_end", F.col("window.end"))
    .drop("window")
).where("zone_id <= 263 and window_end <= '2025-12-31 23:30:00' and window_end >= '2025-01-01 00:30:00'")
# dropoff_counts.printSchema(), dropoff_counts.show(5, truncate=False), dropoff_counts.count()
# dropoff_counts.select(F.min("window_end"), F.max("window_end")).show()

In [16]:
trips_valid = trips.filter(
    F.col("trip_miles").between(0.3, 500) &
    F.col("trip_time").between(200, 20000) &
    F.col("base_passenger_fare").between(1, 1500) &
    F.col("driver_pay").between(1, 1500)
)

dropoff_stats = (
    trips_valid
    .withColumn("zone_id", F.col("PULocationID"))
    .withColumn("window", epoch_to_window_end("dropoff_datetime"))
    .groupBy("zone_id", "window")
    .agg(
        F.mean('trip_time').alias('avg_trip_time'),
        F.stddev('trip_time').alias('std_trip_time'),
        # Price
        F.mean('base_passenger_fare').alias('avg_fare'),
        F.stddev('base_passenger_fare').alias('std_fare'),
        F.mean('driver_pay').alias('avg_driver_pay'),
        F.stddev('driver_pay').alias('std_driver_pay'),
        F.mean('tips').alias('avg_tips'),
        F.stddev('tips').alias('std_tips'),
        F.mean('bcf').alias('avg_bcf'),
        F.stddev('bcf').alias('std_bcf'),
        F.mean('tolls').alias('avg_tolls'),
        F.stddev('tolls').alias('std_tolls'),
        F.mean('congestion_surcharge').alias('avg_congestion_surcharge'),
        F.stddev('congestion_surcharge').alias('std_congestion_surcharge'),
        F.mean('airport_fee').alias('avg_airport_fee'),
        F.stddev('airport_fee').alias('std_airport_fee'),
        F.mean('sales_tax').alias('avg_sales_tax'),
        F.stddev('sales_tax').alias('std_sales_tax'),
        # Distance
        F.mean('trip_miles').alias('avg_distance'),
        F.stddev('trip_miles').alias('std_distance'),
    )
    .withColumn("window_end", F.col("window.end"))
    .drop("window")
).where("zone_id <= 263 and window_end <= '2025-12-31 23:30:00' and window_end >= '2025-01-01 00:30:00'")
# dropoff_stats.printSchema(), dropoff_stats.show(5, truncate=False), dropoff_stats.count()
# dropoff_stats.select(F.min("window_end"), F.max("window_end")).show()

In [17]:
# ─── 2.6 Temporal Features ────────────────────────────────────────────────────
# These are derived from window_end on the final Gold table — no extra join needed.
# We define a helper that adds them post-join.
US_HOLIDAYS = [
    # ===== 2022 =====
    "2022-01-01","2022-01-17","2022-02-21","2022-05-30","2022-06-19",
    "2022-07-04","2022-09-05","2022-10-10","2022-11-11","2022-11-24","2022-12-25",

    # ===== 2023 =====
    "2023-01-01","2023-01-16","2023-02-20","2023-05-29","2023-06-19",
    "2023-07-04","2023-09-04","2023-10-09","2023-11-11","2023-11-23","2023-12-25",

    # ===== 2024 =====
    "2024-01-01","2024-01-15","2024-02-19","2024-05-27","2024-06-19",
    "2024-07-04","2024-09-02","2024-10-14","2024-11-11","2024-11-28","2024-12-25",

    # ===== 2025 =====
    "2025-01-01","2025-01-20","2025-02-17","2025-05-26","2025-06-19",
    "2025-07-04","2025-09-01","2025-10-13","2025-11-11","2025-11-27","2025-12-25",
]

def add_temporal_features(df):

    df = df.withColumn("date", F.to_date("window_end"))

    # ===== BASIC TIME FEATURES =====
    df = (
        df
        .withColumn("hour", F.hour("window_end"))
        .withColumn("minute", F.minute("window_end"))
        .withColumn("day_of_week", F.dayofweek("window_end"))  # 1=Sun
        .withColumn("month", F.month("window_end"))
        .withColumn("week_of_year", F.weekofyear("window_end"))
        .withColumn("year", F.year("window_end"))
        .withColumn("is_weekend", F.col("day_of_week").isin(1,7).cast("int"))
    )

    # ===== HOLIDAY =====
    df = df.withColumn(
        "is_holiday",
        F.col("date").isin(US_HOLIDAYS).cast("int")
    )

    # ===== 5-MIN SLOT =====
    df = df.withColumn(
        "slot_5m",
        F.floor(F.col("minute") / 5) + 1  # +1 to make it 1-indexed
    )

    # ===== CYCLICAL ENCODING =====

    # 1. 5-min slot (1–12)
    df = df.withColumn(
        "slot_5m_sin",
        F.sin(2 * 3.1415926 * F.col("slot_5m") / 12)
    ).withColumn(
        "slot_5m_cos",
        F.cos(2 * 3.1415926 * F.col("slot_5m") / 12)
    )

    # 2. Day of week (1–7)
    df = df.withColumn(
        "dow_sin",
        F.sin(2 * 3.1415926 * F.col("day_of_week") / 7)
    ).withColumn(
        "dow_cos",
        F.cos(2 * 3.1415926 * F.col("day_of_week") / 7)
    )
    
    df = df.withColumn(
        "hou_sin",
        F.sin(2 * 3.1415926 * F.col("hour") / 24)
    ).withColumn(
        "hou_cos",
        F.cos(2 * 3.1415926 * F.col("hour") / 24)
    )

    # 3. Week of year (1–52)
    df = df.withColumn(
        "woy_sin",
        F.sin(2 * 3.1415926 * F.col("week_of_year") / 52)
    ).withColumn(
        "woy_cos",
        F.cos(2 * 3.1415926 * F.col("week_of_year") / 52)
    )
    
    # 4. Month (1–12)
    df = df.withColumn(
        "mon_sin",
        F.sin(2 * 3.1415926 * F.col("month") / 12)
    ).withColumn(
        "mon_cos",
        F.cos(2 * 3.1415926 * F.col("month") / 12)
    )
    
    return df.drop("date", "slot_5m", "hour", "minute", "day_of_week", "month", "week_of_year")  # drop intermediate columns

print('Temporal feature helper defined')

Temporal feature helper defined


In [18]:
# ─── 2.7 Spatial Feature: Neighbor Requests ───────────────────────────────────
# neighbor_requests_30m = sum of requests from adjacent zones in the same window
# We use the TLC taxi zone adjacency list (simplified as a broadcast dictionary).

# !! Replace with actual adjacency data from TLC zone shapefile if available !!
# Below is a small example; in production load the full 263-zone adjacency map.
ZONE_NEIGHBORS = {
    1: [2, 3, 4], 2: [1, 7, 8, 30], 3: [1, 4, 5, 32], 4: [1, 3, 79, 224], 5: [99, 84, 109], 6: [221, 214], 7: [2, 8, 179, 193], 8: [7, 179, 223], 9: [16, 73, 192], 10: [216, 218], 11: [21, 22, 67], 12: [13, 88], 13: [12, 88, 261], 14: [67, 227], 15: [171, 252], 16: [9, 64, 175], 17: [49, 225], 18: [136, 241], 19: [64, 101], 20: [31, 18], 21: [11, 22, 123], 22: [11, 21, 67, 123], 23: [156, 187], 24: [41, 151], 25: [97, 65], 26: [133, 227], 27: [201], 28: [130, 134], 29: [150, 210], 30: [2], 31: [20, 32], 32: [3, 31, 174], 33: [65, 66], 34: [66, 217], 35: [77, 72], 36: [37, 80], 37: [36, 225], 38: [139, 205], 39: [91, 155], 40: [106, 195], 41: [24, 166], 42: [152, 116], 43: [236, 239], 44: [204], 45: [232, 209], 46: [], 47: [59, 169], 48: [50, 163], 49: [17, 97], 50: [48, 142], 51: [81, 184], 52: [54], 53: [252], 54: [52, 33], 55: [108], 56: [82, 95], 57: [173], 58: [183], 59: [47, 60], 60: [59, 78], 61: [62, 189], 62: [61, 188], 63: [177], 64: [16, 19], 65: [25, 33], 66: [33, 34], 67: [11, 14, 22], 68: [100, 246], 69: [247, 119], 70: [129, 173], 71: [85, 72], 72: [71, 188], 73: [9, 171], 74: [41, 75], 75: [74, 236], 76: [77], 77: [35, 76], 78: [60, 20], 79: [4, 107], 80: [36, 255], 81: [51, 254], 82: [56, 83], 83: [82, 260], 84: [5, 109], 85: [71, 89], 86: [117], 87: [209, 261], 88: [12, 13], 89: [85, 165], 90: [234, 186], 91: [39, 165], 92: [171, 253], 93: [95, 135], 94: [136, 169], 95: [56, 93], 96: [102, 198], 97: [25, 49], 98: [121, 175], 99: [5, 118], 100: [68, 230], 101: [19, 64], 102: [96, 95], 103: [], 104: [], 105: [], 106: [40, 181], 107: [79, 137], 108: [55, 123], 109: [5, 84, 110], 110: [109, 176], 111: [190, 227], 112: [255], 113: [114, 249], 114: [113, 125], 115: [221, 245], 116: [42, 152], 117: [86, 201], 118: [99, 109], 119: [69], 120: [244], 121: [98, 135], 122: [191, 131], 123: [21, 22, 108], 124: [180], 125: [114, 158], 126: [147, 168], 127: [128, 243], 128: [127, 153], 129: [70, 207], 130: [28, 134], 131: [122, 98], 132: [219], 133: [26, 111], 134: [28, 130], 135: [93, 121], 136: [18, 94], 137: [107, 170], 138: [223, 207], 139: [38, 203], 140: [141, 202], 141: [140, 237], 142: [143, 50], 143: [142, 239], 144: [148, 211], 145: [193, 226], 146: [193, 7], 147: [126, 159], 148: [144, 232], 149: [155, 123], 150: [29], 151: [24, 238], 152: [42, 166], 153: [128], 154: [], 155: [39, 149], 156: [23], 157: [160, 226], 158: [125, 249], 159: [147, 168], 160: [157, 196], 161: [162, 230], 162: [161, 229], 163: [48, 230], 164: [186, 170], 165: [89, 91], 166: [41, 152], 167: [60, 69], 168: [126, 159], 169: [47, 94], 170: [137, 164], 171: [15, 73, 92], 172: [214, 176], 173: [70, 57], 174: [32, 240], 175: [16, 98], 176: [110, 172], 177: [63], 178: [165], 179: [7, 8], 180: [124], 181: [106, 190], 182: [248], 183: [58], 184: [51], 185: [3], 186: [90, 164], 187: [23], 188: [62, 72], 189: [61], 190: [111, 181], 191: [122], 192: [9], 193: [145, 146], 194: [], 195: [40], 196: [160], 197: [134], 198: [96], 199: [], 200: [240], 201: [27, 117], 202: [140], 203: [139], 204: [44], 205: [38], 206: [245], 207: [129, 138], 208: [], 209: [45, 87], 210: [29], 211: [144], 212: [248], 213: [], 214: [6, 172], 215: [218], 216: [10], 217: [34], 218: [10, 215], 219: [132], 220: [200], 221: [6, 115], 222: [], 223: [8, 138], 224: [4], 225: [17, 37], 226: [145, 157], 227: [14, 26, 111], 228: [], 229: [162], 230: [100, 161, 163], 231: [211], 232: [45, 148], 233: [170], 234: [90], 235: [], 236: [43, 75], 237: [141], 238: [151], 239: [43, 143], 240: [174, 200], 241: [18], 242: [], 243: [127], 244: [120], 245: [115, 206], 246: [68], 247: [69], 248: [182, 212], 249: [113, 158], 250: [], 251: [], 252: [15, 53], 253: [92], 254: [81], 255: [80, 112], 256: [], 257: [], 258: [], 259: [], 260: [83], 261: [13, 87], 262: [], 263: []
}

# Build adjacency as a Spark dataframe for a scalable join
adjacency_rows = [
    (zone, neighbor)
    for zone, neighbors in ZONE_NEIGHBORS.items()
    for neighbor in neighbors
]

adj_schema = StructType([
    StructField('zone_id',          IntegerType(), False),
    StructField('neighbor_zone_id', IntegerType(), False),
])

adj_df = spark.createDataFrame(adjacency_rows, schema=adj_schema)

def compute_neighbor_requests(demand_df, adj_df):
    neighbor_demand = (
        demand_df
        .join(
            adj_df,
            demand_df['zone_id'] == adj_df['neighbor_zone_id'],
            'left'  
        )
        .groupBy(demand_df['zone_id'], 'window_end')
        .agg(
            F.sum('requests_30m').alias('neighbor_requests_30m'),
            F.sum('pickup_30m').alias('neighbor_pickup_30m'),
            F.sum('dropoff_30m').alias('neighbor_dropoff_30m'),
        )
        .fillna(0)   # zone không có neighbor -> 0
        .withColumn(
            "neighbor_imbalance",
            (F.col("neighbor_requests_30m") + 49.9) /
            (F.col("neighbor_dropoff_30m") - F.col("neighbor_pickup_30m") + 49.9)
        )
    )

    return neighbor_demand

print('Neighbor spatial feature helper defined')

Neighbor spatial feature helper defined


In [ ]:
# ─── 2.8 Assemble Gold Feature Table (pre-lag) ────────────────────────────────

# Align 5-minute features to the 30-minute window grid.
# Strategy: the 5m slice ending closest to window_end is the 'most recent 5m' slice.
# We join demand_5m_raw where window_end_5m == window_end.

gold = (
    demand
    .join(pickup, ["zone_id", "window_end"], "outer")
    .join(dropoff_counts, ["zone_id", "window_end"], "outer")
    .join(dropoff_stats, ["zone_id", "window_end"], "outer")
)
# Neighbor requests
neighbor_feat = compute_neighbor_requests(
    gold.select(
        'zone_id',
        'window_end',
        'requests_30m',
        'pickup_30m',
        'dropoff_30m'
    ),
    adj_df
)

gold = gold.join(neighbor_feat, ['zone_id', 'window_end'], 'left')
COUNT_COLS = [
    'requests_30m', 'requests_5m',
    'pickup_30m', 'pickup_5m',
    'dropoff_30m', 'dropoff_5m',
    'matched_rp', 'matched_rp_5m',
    'matched_rd', 'matched_pd',
    'wav_req_30m', 'aar_req_30m', 'share_req_30m',
    'wav_match_30m', 'share_match_30m',
    'neighbor_requests_30m',
    'neighbor_pickup_30m',
    'neighbor_dropoff_30m',
    'uber_req', 'lyft_req',
]
STAT_COLS = [
    'pickup_delay_mean', 'pickup_delay_std',
    'avg_trip_time', 'std_trip_time',
    'avg_fare', 'std_fare',
    'avg_driver_pay', 'std_driver_pay',
    'avg_tips', 'std_tips',
    'avg_distance', 'std_distance'
]

COUNT_COLS = [c for c in COUNT_COLS if c in gold.columns]

gold = gold.fillna(0, subset=COUNT_COLS)

w = (
    Window
    .partitionBy("zone_id")
    .orderBy("window_end")
    .rowsBetween(-6, 0)
)
for c in STAT_COLS:
    gold = gold.withColumn(
        c + "_ffill",
        F.last(c, ignorenulls=True).over(w)
    )
for c in STAT_COLS:
    gold = gold.withColumn(
        c,
        F.col(c + "_ffill")
    )
gold = gold.drop(*[c + "_ffill" for c in STAT_COLS])
gold = gold.withColumn("imbalance", (F.col("requests_30m") + 49.9) / (F.col("dropoff_30m") - F.col("pickup_30m") + 49.9))
gold.printSchema(), gold.show(5, truncate=False), gold.count()

root
 |-- zone_id: long (nullable = true)
 |-- window_end: timestamp_ntz (nullable = true)
 |-- requests_30m: long (nullable = true)
 |-- requests_5m: long (nullable = true)
 |-- wav_req_30m: long (nullable = true)
 |-- aar_req_30m: long (nullable = true)
 |-- share_req_30m: long (nullable = true)
 |-- uber_req: long (nullable = true)
 |-- lyft_req: long (nullable = true)
 |-- pickup_30m: long (nullable = true)
 |-- pickup_5m: long (nullable = true)
 |-- pickup_delay_mean: double (nullable = true)
 |-- pickup_delay_std: double (nullable = true)
 |-- matched_rp: long (nullable = true)
 |-- matched_rp_5m: long (nullable = true)
 |-- wav_match_30m: long (nullable = true)
 |-- share_match_30m: long (nullable = true)
 |-- dropoff_30m: long (nullable = true)
 |-- dropoff_5m: long (nullable = true)
 |-- matched_rd: long (nullable = true)
 |-- matched_pd: long (nullable = true)
 |-- avg_trip_time: double (nullable = true)
 |-- std_trip_time: double (nullable = true)
 |-- avg_fare: double (null

(None, None, 26730922)

In [ ]:
# Temporal features
gold = add_temporal_features(gold)

print('Gold table assembled (before lag features)')

Gold table assembled (before lag features)
+-------+-------------------+------------+-----------+-----------+-----------+-------------+--------+--------+----------+---------+-----------------+----------------+----------+-------------+-------------+---------------+-----------+----------+----------+----------+-------------+-------------+--------+--------+--------------+--------------+--------+--------+-------+-------+---------+---------+------------------------+------------------------+---------------+---------------+-------------+-------------+------------+------------+------------------+---------------------+-------------------+--------------------+-------------------+----+----------+----------+---------------------+---------------------+-------------------+-------------------+-------------------+-------------------+-------------------+------------------+-------------------+------------------+
|zone_id|window_end         |requests_30m|requests_5m|wav_req_30m|aar_req_30m|share_req_30m|u

## Section 2.5 - GOLD_ALL

In [11]:
# ─── Window Configuration ─────────────────────────────────────────────────────
WINDOW_SECONDS = 30 * 60   # 30 min in seconds
SLIDE_SECONDS  =  5 * 60   # 5 min  in seconds

def epoch_to_window_end(ts_col: str, window_s: int = WINDOW_SECONDS, slide_s: int = SLIDE_SECONDS):
    """
    Assign each timestamp to the next (future) window boundary.
    window_end = ceil(ts / slide) * slide  — snaps to the next slide boundary.
    Spark's window() function handles this natively.
    """
    return F.window(F.col(ts_col), f'{window_s} seconds', f'{slide_s} seconds')

flag = lambda c: F.when(F.col(c) == "Y", 1).otherwise(0)
is_uber = F.when(F.col("hvfhs_license_num") == "HV0003", 1).otherwise(0)
is_lyft = F.when(F.col("hvfhs_license_num") == "HV0005", 1).otherwise(0)

print('Window helpers defined')

Window helpers defined


In [12]:
demand_all = (
    trips
    # .withColumn("zone_id", F.col("PULocationID"))
    .withColumn("window", epoch_to_window_end("request_datetime"))
    .groupBy("window")
    .agg(
        # volume
        F.count("*").alias("requests_30m"),
        F.sum(
            F.when(
                F.col("request_datetime") >= F.col("window.end") - F.expr(f"INTERVAL {SLIDE_SECONDS} seconds"),
                1
            ).otherwise(0)
        ).alias("requests_5m"),

        # WAV / AAR (request-time)
        F.sum(flag("wav_request_flag")).alias("wav_req_30m"),
        F.sum(flag("access_a_ride_flag")).alias("aar_req_30m"),
        F.sum(flag("share_request_flag")).alias("share_req_30m"),

        # platform share
        F.sum(is_uber).alias("uber_req"),
        F.sum(is_lyft).alias("lyft_req"),
    )
    .withColumn("window_end", F.col("window.end"))
    .drop("window")
).where("window_end <= '2025-12-31 23:30:00' and window_end >= '2025-01-01 00:30:00'")
pickup_all = (
    trips
    # .withColumn("zone_id", F.col("PULocationID"))
    .withColumn("window", epoch_to_window_end("pickup_datetime"))
    .withColumn(
        "pickup_delay_s",
        F.unix_timestamp("pickup_datetime") - F.unix_timestamp("request_datetime")
    )
    .groupBy("window")
    .agg(
        # volume
        F.count("*").alias("pickup_30m"),
        F.sum(
            F.when(
                F.col("pickup_datetime") >= F.col("window.end") - F.expr(f"INTERVAL {SLIDE_SECONDS} seconds"),
                1
            )
        ).alias("pickup_5m"),

        # delay
        F.mean("pickup_delay_s").alias("pickup_delay_mean"),
        F.stddev("pickup_delay_s").alias("pickup_delay_std"),
        
        # matching (proxy)
        F.sum(
            F.when(col('request_datetime') >= col('window.start') , 1).otherwise(0)
        ).alias('matched_rp'),
        
        F.sum(
            F.when(((col('request_datetime') >= col('window.start') )
                   & (F.col("pickup_datetime") >= (F.col("window.end") - F.expr(f"INTERVAL {SLIDE_SECONDS} seconds")))), 1).otherwise(0)
        ).alias('matched_rp_5m'),

        # WAV match
        F.sum(flag("wav_match_flag")).alias("wav_match_30m"),
        F.sum(flag("share_match_flag")).alias("share_match_30m"),

    )
    .withColumn("window_end", F.col("window.end"))
    .drop("window")
).where("window_end <= '2025-12-31 23:30:00' and window_end >= '2025-01-01 00:30:00'")


In [13]:
dropoff_counts_all = (
    trips
    # .withColumn("zone_id", F.col("DOLocationID"))
    .withColumn("window", epoch_to_window_end("dropoff_datetime"))
    .groupBy("window")
    .agg(
        F.count("*").alias("dropoff_30m"),
        F.sum(
            F.when(
                F.col("dropoff_datetime") >= F.col("window.end") - F.expr(f"INTERVAL {SLIDE_SECONDS} seconds"),
                1
            )
        ).alias("dropoff_5m"),
        
        F.sum(
            F.when(((col('request_datetime') >= col('window.start')) & (col('DOLocationID') == col('PULocationID'))), 1).otherwise(0)
        ).alias('matched_rd'),
        
        F.sum(
            F.when(((col('pickup_datetime') >= col('window.start')) & (col('DOLocationID') == col('PULocationID'))), 1).otherwise(0)
        ).alias('matched_pd'), )
    .withColumn("window_end", F.col("window.end"))
    .drop("window")
).where("window_end <= '2025-12-31 23:30:00' and window_end >= '2025-01-01 00:30:00'")

In [15]:
trips_valid = trips.filter(
    F.col("trip_miles").between(0.3, 500) &
    F.col("trip_time").between(200, 20000) &
    F.col("base_passenger_fare").between(1, 1500) &
    F.col("driver_pay").between(1, 1500)
)
dropoff_stats_all = (
    trips_valid
    # .withColumn("zone_id", F.col("PULocationID"))
    .withColumn("window", epoch_to_window_end("dropoff_datetime"))
    .groupBy("window")
    .agg(
        F.mean('trip_time').alias('avg_trip_time'),
        F.stddev('trip_time').alias('std_trip_time'),
        # Price
        F.mean('base_passenger_fare').alias('avg_fare'),
        F.stddev('base_passenger_fare').alias('std_fare'),
        F.mean('driver_pay').alias('avg_driver_pay'),
        F.stddev('driver_pay').alias('std_driver_pay'),
        F.mean('tips').alias('avg_tips'),
        F.stddev('tips').alias('std_tips'),
        F.mean('bcf').alias('avg_bcf'),
        F.stddev('bcf').alias('std_bcf'),
        F.mean('tolls').alias('avg_tolls'),
        F.stddev('tolls').alias('std_tolls'),
        F.mean('congestion_surcharge').alias('avg_congestion_surcharge'),
        F.stddev('congestion_surcharge').alias('std_congestion_surcharge'),
        F.mean('airport_fee').alias('avg_airport_fee'),
        F.stddev('airport_fee').alias('std_airport_fee'),
        F.mean('sales_tax').alias('avg_sales_tax'),
        F.stddev('sales_tax').alias('std_sales_tax'),
        # Distance
        F.mean('trip_miles').alias('avg_distance'),
        F.stddev('trip_miles').alias('std_distance'),
    )
    .withColumn("window_end", F.col("window.end"))
    .drop("window")
).where("window_end <= '2025-12-31 23:30:00' and window_end >= '2025-01-01 00:30:00'")

In [16]:
# ─── 2.6 Temporal Features ────────────────────────────────────────────────────
# These are derived from window_end on the final Gold table — no extra join needed.
# We define a helper that adds them post-join.
US_HOLIDAYS = [
    # ===== 2022 =====
    "2022-01-01","2022-01-17","2022-02-21","2022-05-30","2022-06-19",
    "2022-07-04","2022-09-05","2022-10-10","2022-11-11","2022-11-24","2022-12-25",

    # ===== 2023 =====
    "2023-01-01","2023-01-16","2023-02-20","2023-05-29","2023-06-19",
    "2023-07-04","2023-09-04","2023-10-09","2023-11-11","2023-11-23","2023-12-25",

    # ===== 2024 =====
    "2024-01-01","2024-01-15","2024-02-19","2024-05-27","2024-06-19",
    "2024-07-04","2024-09-02","2024-10-14","2024-11-11","2024-11-28","2024-12-25",

    # ===== 2025 =====
    "2025-01-01","2025-01-20","2025-02-17","2025-05-26","2025-06-19",
    "2025-07-04","2025-09-01","2025-10-13","2025-11-11","2025-11-27","2025-12-25",
]

def add_temporal_features(df):

    df = df.withColumn("date", F.to_date("window_end"))

    # ===== BASIC TIME FEATURES =====
    df = (
        df
        .withColumn("hour", F.hour("window_end"))
        .withColumn("minute", F.minute("window_end"))
        .withColumn("day_of_week", F.dayofweek("window_end"))  # 1=Sun
        .withColumn("month", F.month("window_end"))
        .withColumn("week_of_year", F.weekofyear("window_end"))
        .withColumn("year", F.year("window_end"))
        .withColumn("is_weekend", F.col("day_of_week").isin(1,7).cast("int"))
    )

    # ===== HOLIDAY =====
    df = df.withColumn(
        "is_holiday",
        F.col("date").isin(US_HOLIDAYS).cast("int")
    )

    # ===== 5-MIN SLOT =====
    df = df.withColumn(
        "slot_5m",
        F.floor(F.col("minute") / 5) + 1  # +1 to make it 1-indexed
    )

    # ===== CYCLICAL ENCODING =====

    # 1. 5-min slot (1–12)
    df = df.withColumn(
        "slot_5m_sin",
        F.sin(2 * 3.1415926 * F.col("slot_5m") / 12)
    ).withColumn(
        "slot_5m_cos",
        F.cos(2 * 3.1415926 * F.col("slot_5m") / 12)
    )

    # 2. Day of week (1–7)
    df = df.withColumn(
        "dow_sin",
        F.sin(2 * 3.1415926 * F.col("day_of_week") / 7)
    ).withColumn(
        "dow_cos",
        F.cos(2 * 3.1415926 * F.col("day_of_week") / 7)
    )
    
    df = df.withColumn(
        "hou_sin",
        F.sin(2 * 3.1415926 * F.col("hour") / 24)
    ).withColumn(
        "hou_cos",
        F.cos(2 * 3.1415926 * F.col("hour") / 24)
    )

    # 3. Week of year (1–52)
    df = df.withColumn(
        "woy_sin",
        F.sin(2 * 3.1415926 * F.col("week_of_year") / 52)
    ).withColumn(
        "woy_cos",
        F.cos(2 * 3.1415926 * F.col("week_of_year") / 52)
    )
    
    # 4. Month (1–12)
    df = df.withColumn(
        "mon_sin",
        F.sin(2 * 3.1415926 * F.col("month") / 12)
    ).withColumn(
        "mon_cos",
        F.cos(2 * 3.1415926 * F.col("month") / 12)
    )
    
    return df.drop("date", "slot_5m", "hour", "minute", "day_of_week", "month", "week_of_year")  # drop intermediate columns

print('Temporal feature helper defined')

Temporal feature helper defined


In [18]:
# ─── 2.8 Assemble Gold Feature Table (pre-lag) ────────────────────────────────

# Align 5-minute features to the 30-minute window grid.
# Strategy: the 5m slice ending closest to window_end is the 'most recent 5m' slice.
# We join demand_5m_raw where window_end_5m == window_end.

gold_all = (
    demand_all
    .join(pickup_all, [ "window_end"], "outer")
    .join(dropoff_counts_all, [ "window_end"], "outer")
    .join(dropoff_stats_all, [ "window_end"], "outer")
)

COUNT_COLS = [
    'requests_30m', 'requests_5m',
    'pickup_30m', 'pickup_5m',
    'dropoff_30m', 'dropoff_5m',
    'matched_rp', 'matched_rp_5m',
    'matched_rd', 'matched_pd',
    'wav_req_30m', 'aar_req_30m', 'share_req_30m',
    'wav_match_30m', 'share_match_30m',
    'uber_req', 'lyft_req',
]
STAT_COLS = [
    'pickup_delay_mean', 'pickup_delay_std',
    'avg_trip_time', 'std_trip_time',
    'avg_fare', 'std_fare',
    'avg_driver_pay', 'std_driver_pay',
    'avg_tips', 'std_tips',
    'avg_distance', 'std_distance'
]

COUNT_COLS = [c for c in COUNT_COLS if c in gold_all.columns]

gold_all = gold_all.fillna(0, subset=COUNT_COLS)

w = (
    Window
    .orderBy("window_end")
    .rowsBetween(-6, 0)
)
for c in STAT_COLS:
    gold_all = gold_all.withColumn(
        c + "_ffill",
        F.last(c, ignorenulls=True).over(w)
    )
for c in STAT_COLS:
    gold_all = gold_all.withColumn(
        c,
        F.col(c + "_ffill")
    )
gold_all = gold_all.drop(*[c + "_ffill" for c in STAT_COLS])
gold_all = gold_all.withColumn("imbalance", (F.col("requests_30m") + 49.9) / (F.col("dropoff_30m") - F.col("pickup_30m") + 49.9))
gold_all.printSchema(), gold_all.show(5, truncate=False), gold_all.count()

root
 |-- window_end: timestamp_ntz (nullable = true)
 |-- requests_30m: long (nullable = true)
 |-- requests_5m: long (nullable = true)
 |-- wav_req_30m: long (nullable = true)
 |-- aar_req_30m: long (nullable = true)
 |-- share_req_30m: long (nullable = true)
 |-- uber_req: long (nullable = true)
 |-- lyft_req: long (nullable = true)
 |-- pickup_30m: long (nullable = true)
 |-- pickup_5m: long (nullable = true)
 |-- pickup_delay_mean: double (nullable = true)
 |-- pickup_delay_std: double (nullable = true)
 |-- matched_rp: long (nullable = true)
 |-- matched_rp_5m: long (nullable = true)
 |-- wav_match_30m: long (nullable = true)
 |-- share_match_30m: long (nullable = true)
 |-- dropoff_30m: long (nullable = true)
 |-- dropoff_5m: long (nullable = true)
 |-- matched_rd: long (nullable = true)
 |-- matched_pd: long (nullable = true)
 |-- avg_trip_time: double (nullable = true)
 |-- std_trip_time: double (nullable = true)
 |-- avg_fare: double (nullable = true)
 |-- std_fare: double (n

(None, None, 105108)

In [20]:
gold_all = add_temporal_features(gold_all)
gold_all.cache()

DataFrame[window_end: timestamp_ntz, requests_30m: bigint, requests_5m: bigint, wav_req_30m: bigint, aar_req_30m: bigint, share_req_30m: bigint, uber_req: bigint, lyft_req: bigint, pickup_30m: bigint, pickup_5m: bigint, pickup_delay_mean: double, pickup_delay_std: double, matched_rp: bigint, matched_rp_5m: bigint, wav_match_30m: bigint, share_match_30m: bigint, dropoff_30m: bigint, dropoff_5m: bigint, matched_rd: bigint, matched_pd: bigint, avg_trip_time: double, std_trip_time: double, avg_fare: double, std_fare: double, avg_driver_pay: double, std_driver_pay: double, avg_tips: double, std_tips: double, avg_bcf: double, std_bcf: double, avg_tolls: double, std_tolls: double, avg_congestion_surcharge: double, std_congestion_surcharge: double, avg_airport_fee: double, std_airport_fee: double, avg_sales_tax: double, std_sales_tax: double, avg_distance: double, std_distance: double, imbalance: double, year: int, is_weekend: int, is_holiday: int, slot_5m_sin: double, slot_5m_cos: double, dow

---
## Section 3 — Label Construction, LAG


In [22]:
# ─── 3.1 Compute Future Demand and Supply ─────────────────────────────────────
# We shift the demand_raw and supply_30m tables by one 30-minute step forward.
# future_window_start = current window_end
# future_window_end   = window_end + 30 minutes

OFFSET_S = WINDOW_SECONDS  # shift forward by one full 30m window

future_demand = (
    demand_raw
    .withColumn(
        'join_key',
        F.from_unixtime(
            F.unix_timestamp('window_end') - OFFSET_S
        ).cast(TimestampType())
    )
    .select(
        'zone_id',
        F.col('join_key').alias('window_end'),
        F.col('requests_30m').alias('future_requests_30m')
    )
)

future_supply = (
    supply_30m
    .withColumn(
        'join_key',
        F.from_unixtime(
            F.unix_timestamp('window_end') - OFFSET_S
        ).cast(TimestampType())
    )
    .select(
        'zone_id',
        F.col('join_key').alias('window_end'),
        F.col('completed_30m').alias('future_completed_30m')
    )
)

print('Future demand and supply computed')

NameError: name 'demand_raw' is not defined

In [ ]:
# ─── 3.2 Compute supply_demand_ratio ──────────────────────────────────────────

gold_labeled = (
    gold
    .join(future_demand, ['zone_id', 'window_end'], 'left')
    .join(future_supply, ['zone_id', 'window_end'], 'left')
    .fillna(0, subset=['future_requests_30m', 'future_completed_30m'])
    .withColumn(
        'supply_demand_ratio',
        F.when(
            F.col('future_requests_30m') == 0,
            # No demand → ratio is 1.0 (neutral; driver is not needed)
            F.lit(1.0)
        ).otherwise(
            F.col('future_completed_30m') / F.col('future_requests_30m')
        ).cast(DoubleType())
    )
    # Drop rows where we have no future data (last window)
    .filter(F.col('future_requests_30m').isNotNull())
)

print('supply_demand_ratio computed')

supply_demand_ratio computed


In [ ]:
# ─── 3.3 Label Strategies ─────────────────────────────────────────────────────

# --- Compute quantiles for informed thresholds ---
ratio_quantiles = (
    gold_labeled
    .stat.approxQuantile('supply_demand_ratio', [0.25, 0.5, 0.75], 0.01)
)
Q1, Q2, Q3 = ratio_quantiles
print(f'Quantiles → Q1={Q1:.3f}  Q2={Q2:.3f}  Q3={Q3:.3f}')

# --- Binary: under-supplied vs balanced/surplus ---
gold_labeled = gold_labeled.withColumn(
    'label_binary',
    F.when(F.col('supply_demand_ratio') < 1.0, 0).otherwise(1).cast(IntegerType())
)

# --- 3-class: severe shortage / mild shortage / surplus ---
gold_labeled = gold_labeled.withColumn(
    'label_3class',
    F.when(F.col('supply_demand_ratio') < Q1,                          F.lit(0))  # severe shortage
     .when(F.col('supply_demand_ratio').between(Q1, 1.0),              F.lit(1))  # mild shortage
     .otherwise(                                                        F.lit(2))  # surplus
     .cast(IntegerType())
)

# --- 4-class: quantile-based ---
gold_labeled = gold_labeled.withColumn(
    'label_4class',
    F.when(F.col('supply_demand_ratio') < Q1,                          F.lit(0))  # very short
     .when(F.col('supply_demand_ratio').between(Q1, Q2),               F.lit(1))  # short
     .when(F.col('supply_demand_ratio').between(Q2, Q3),               F.lit(2))  # balanced
     .otherwise(                                                        F.lit(3))  # surplus
     .cast(IntegerType())
)

gold_labeled = gold_labeled.cache()
print(f'Gold labeled rows: {gold_labeled.count():,}')
gold_labeled.select(
    'zone_id', 'window_end',
    'supply_demand_ratio',
    'label_binary', 'label_3class', 'label_4class'
).show(10)

ValueError: not enough values to unpack (expected 3, got 0)

---
## Section 4 — Feature-by-Feature Validation

For each feature we:
1. Pull a sample to Pandas
2. Plot distribution + time series + correlation with target
3. Write an evaluation note

> We sample from **one representative zone** (zone 161 = Midtown Center) for time-series clarity.

In [ ]:
# ─── Sample to Pandas ─────────────────────────────────────────────────────────
SAMPLE_ZONE = 161
SAMPLE_SIZE = 5000

ALL_FEATURES = [
    'requests_30m', 'requests_5m', 'lag_1_requests', 'lag_6_requests',
    'completed_30m', 'completed_5m',
    'matched_30m', 'matched_5m', 'pickup_delay_mean', 'pickup_delay_std',
    'inflow_30m', 'outflow_30m',
    'avg_speed', 'std_speed', 'avg_trip_time', 'std_trip_time',
    'avg_fare', 'std_fare', 'avg_driver_pay',
    'avg_distance', 'std_distance',
    'hour_of_day', 'day_of_week', 'is_weekend',
    'neighbor_requests_30m',
]

LABEL_COLS = ['supply_demand_ratio', 'label_binary', 'label_3class', 'label_4class']

# Full-dataset sample (for global distributions)
pdf_all = (
    gold_labeled
    .select(['zone_id', 'window_end'] + ALL_FEATURES + LABEL_COLS)
    .dropna(subset=['supply_demand_ratio'])
    .sample(fraction=min(1.0, SAMPLE_SIZE * 10 / gold_labeled.count()), seed=42)
    .limit(SAMPLE_SIZE * 10)
    .toPandas()
)

# Zone-specific time series
pdf_zone = (
    gold_labeled
    .filter(F.col('zone_id') == SAMPLE_ZONE)
    .select(['window_end'] + ALL_FEATURES + LABEL_COLS)
    .orderBy('window_end')
    .toPandas()
)

pdf_all['window_end']  = pd.to_datetime(pdf_all['window_end'])
pdf_zone['window_end'] = pd.to_datetime(pdf_zone['window_end'])

print(f'Global sample: {len(pdf_all):,} rows')
print(f'Zone {SAMPLE_ZONE} sample: {len(pdf_zone):,} rows')
pdf_all.describe()

In [ ]:
# ─── Validation Helper ────────────────────────────────────────────────────────

def validate_feature(feat_name, pdf_global, pdf_ts, evaluation_text):
    """
    Standard 3-panel validation plot for a single feature.
    """
    fig, axes = plt.subplots(1, 3, figsize=(18, 4))
    fig.suptitle(f'Feature Validation: {feat_name}', fontsize=14, fontweight='bold')

    col = pdf_global[feat_name].dropna()
    target = pdf_global['supply_demand_ratio'].dropna()
    valid_mask = pdf_global[feat_name].notna() & pdf_global['supply_demand_ratio'].notna()

    # Panel 1: Distribution
    ax = axes[0]
    ax.hist(col, bins=50, color=PALETTE[0], edgecolor='white', linewidth=0.3)
    ax.set_title('Distribution')
    ax.set_xlabel(feat_name)
    ax.set_ylabel('Count')

    # Panel 2: Time series (zone-level)
    ax = axes[1]
    ts_col = pdf_ts[feat_name].dropna()
    ax.plot(pdf_ts['window_end'].iloc[ts_col.index],
            ts_col.values, color=PALETTE[1], linewidth=0.8, alpha=0.9)
    ax.set_title(f'Time Series — Zone {SAMPLE_ZONE}')
    ax.set_xlabel('Time')
    ax.set_ylabel(feat_name)
    ax.tick_params(axis='x', rotation=30)

    # Panel 3: Scatter correlation with target
    ax = axes[2]
    x = pdf_global.loc[valid_mask, feat_name]
    y = pdf_global.loc[valid_mask, 'supply_demand_ratio']
    corr = x.corr(y)
    ax.scatter(x, y, alpha=0.15, s=5, color=PALETTE[2])
    ax.set_title(f'Correlation with Target (r={corr:.3f})')
    ax.set_xlabel(feat_name)
    ax.set_ylabel('supply_demand_ratio')

    plt.tight_layout()
    plt.show()

    print(f'\n📋 Evaluation — {feat_name}')
    print(f'   Missing: {pdf_global[feat_name].isna().mean()*100:.1f}%')
    print(f'   Pearson r with target: {corr:.4f}')
    print(f'   {evaluation_text}\n')

print('Validation helper ready')

In [ ]:
# ─── 4.1 requests_30m ─────────────────────────────────────────────────────────
validate_feature(
    'requests_30m', pdf_all, pdf_zone,
    """USEFUL: Core demand signal. Higher requests without matching supply increase imbalance.
    Expected moderate negative correlation with supply_demand_ratio.
    Low redundancy — this is the direct demand input."""
)

In [ ]:
# ─── 4.2 requests_5m ──────────────────────────────────────────────────────────
validate_feature(
    'requests_5m', pdf_all, pdf_zone,
    """USEFUL (conditional): Captures demand momentum in the most recent slice.
    Complements requests_30m but may be partially redundant if the window is stable.
    Check VIF against requests_30m — consider keeping only if VIF < 5."""
)

In [ ]:
# ─── 4.3 lag_1_requests ───────────────────────────────────────────────────────
validate_feature(
    'lag_1_requests', pdf_all, pdf_zone,
    """USEFUL: Previous-window demand. Captures short-term autocorrelation.
    Critical for the model to understand whether demand is increasing or decreasing.
    Typically high positive correlation with requests_30m — VIF check recommended."""
)

In [ ]:
# ─── 4.4 lag_6_requests ───────────────────────────────────────────────────────
validate_feature(
    'lag_6_requests', pdf_all, pdf_zone,
    """MODERATELY USEFUL: 30-minute-ago demand. Captures same-period trend.
    Less correlated with requests_30m than lag_1. Adds a longer-horizon context.
    Keep if correlation with target differs noticeably from lag_1."""
)

In [ ]:
# ─── 4.5 completed_30m ────────────────────────────────────────────────────────
validate_feature(
    'completed_30m', pdf_all, pdf_zone,
    """USEFUL: Core supply signal. More completions = more drivers available in zone.
    Expected positive correlation with ratio. Primary supply proxy."""
)

In [ ]:
# ─── 4.6 completed_5m ─────────────────────────────────────────────────────────
validate_feature(
    'completed_5m', pdf_all, pdf_zone,
    """MODERATELY USEFUL: Recent supply momentum. Redundant if completed_30m is stable.
    May offer leading-edge signal when supply is rapidly changing."""
)

In [ ]:
# ─── 4.7 matched_30m ──────────────────────────────────────────────────────────
validate_feature(
    'matched_30m', pdf_all, pdf_zone,
    """USEFUL: Reflects actual marketplace matching efficiency.
    Complements completed_30m by focusing on matching (pickup confirmed) vs. supply availability.
    High collinearity with completed_30m expected — VIF check recommended."""
)

In [ ]:
# ─── 4.8 matched_5m ───────────────────────────────────────────────────────────
validate_feature(
    'matched_5m', pdf_all, pdf_zone,
    """CONDITIONALLY USEFUL: Same rationale as completed_5m.
    Drop if matched_30m VIF is already high — may be redundant in tree-based models."""
)

In [ ]:
# ─── 4.9 pickup_delay_mean ────────────────────────────────────────────────────
validate_feature(
    'pickup_delay_mean', pdf_all, pdf_zone,
    """VERY USEFUL: Direct proxy for supply-demand imbalance.
    Higher delay → demand exceeds supply → lower ratio.
    Expected strong negative correlation with supply_demand_ratio.
    One of the most informative features."""
)

In [ ]:
# ─── 4.10 pickup_delay_std ────────────────────────────────────────────────────
validate_feature(
    'pickup_delay_std', pdf_all, pdf_zone,
    """USEFUL: Captures variability in matching quality.
    High std with high mean delay = chaotic supply. Adds information beyond the mean.
    Keep — low redundancy with pickup_delay_mean."""
)

In [ ]:
# ─── 4.11 inflow_30m ──────────────────────────────────────────────────────────
validate_feature(
    'inflow_30m', pdf_all, pdf_zone,
    """USEFUL: Drivers arriving in the zone (completing trips elsewhere).
    Leading indicator of future supply. Positive correlation with ratio expected.
    Distinct signal from completed_30m (which tracks departures)."""
)

In [ ]:
# ─── 4.12 outflow_30m ─────────────────────────────────────────────────────────
validate_feature(
    'outflow_30m', pdf_all, pdf_zone,
    """USEFUL: Drivers leaving the zone. Higher outflow → fewer drivers available next.
    Negative correlation with ratio expected.
    inflow - outflow = net flow, a strong imbalance proxy."""
)

In [ ]:
# ─── 4.13 avg_speed ───────────────────────────────────────────────────────────
validate_feature(
    'avg_speed', pdf_all, pdf_zone,
    """MODERATELY USEFUL: Proxy for road congestion. Lower speed = longer trip time = fewer
    available drivers. Correlated with avg_trip_time — pick one if VIF is high.
    Useful during rush-hour modeling."""
)

In [ ]:
# ─── 4.14 std_speed ───────────────────────────────────────────────────────────
validate_feature(
    'std_speed', pdf_all, pdf_zone,
    """MARGINAL: High variance in speed may capture stop-and-go congestion.
    Likely redundant given avg_speed. Consider dropping if feature importance is low."""
)

In [ ]:
# ─── 4.15 avg_trip_time ───────────────────────────────────────────────────────
validate_feature(
    'avg_trip_time', pdf_all, pdf_zone,
    """MODERATELY USEFUL: Longer trips lock drivers out of the zone longer.
    Negatively correlated with supply availability.
    Some overlap with avg_speed — keep both only if VIF < 5."""
)

In [ ]:
# ─── 4.16 std_trip_time ───────────────────────────────────────────────────────
validate_feature(
    'std_trip_time', pdf_all, pdf_zone,
    """MARGINAL: High variance in trip time may signal route instability.
    Low standalone correlation expected. Consider dropping."""
)

In [ ]:
# ─── 4.17 avg_fare ────────────────────────────────────────────────────────────
validate_feature(
    'avg_fare', pdf_all, pdf_zone,
    """USEFUL: Surge pricing correlates with imbalance.
    Higher fares = platform detected shortage = negative ratio expected.
    Strong signal especially during peak events."""
)

In [ ]:
# ─── 4.18 std_fare ────────────────────────────────────────────────────────────
validate_feature(
    'std_fare', pdf_all, pdf_zone,
    """MARGINAL: Fare variability may indicate mixed demand types.
    Low direct correlation with imbalance. Consider dropping."""
)

In [ ]:
# ─── 4.19 avg_driver_pay ──────────────────────────────────────────────────────
validate_feature(
    'avg_driver_pay', pdf_all, pdf_zone,
    """USEFUL: High driver pay signals longer or more valuable trips.
    Also correlates with surge. Positively related to driver incentive to stay in zone.
    Partially redundant with avg_fare — keep if they diverge (e.g., tips differ)."""
)

In [ ]:
# ─── 4.20 avg_distance ────────────────────────────────────────────────────────
validate_feature(
    'avg_distance', pdf_all, pdf_zone,
    """MODERATELY USEFUL: Longer trips = drivers exit zone for longer periods.
    Negatively correlated with near-term supply.
    Some overlap with avg_trip_time — VIF check recommended."""
)

In [ ]:
# ─── 4.21 std_distance ────────────────────────────────────────────────────────
validate_feature(
    'std_distance', pdf_all, pdf_zone,
    """MARGINAL: High variance in distance = heterogeneous trip mix.
    Low direct correlation expected. Candidate for removal."""
)

In [ ]:
# ─── 4.22 hour_of_day ─────────────────────────────────────────────────────────
# For temporal features, use a box plot instead of scatter
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
fig.suptitle('Feature Validation: hour_of_day', fontsize=14, fontweight='bold')

ax = axes[0]
pdf_all.boxplot(column='supply_demand_ratio', by='hour_of_day', ax=ax)
ax.set_title('Ratio by Hour of Day')
ax.set_xlabel('hour_of_day')
ax.set_ylabel('supply_demand_ratio')

ax = axes[1]
hourly_avg = pdf_all.groupby('hour_of_day')['requests_30m'].mean()
ax.bar(hourly_avg.index, hourly_avg.values, color=PALETTE[3])
ax.set_title('Average requests_30m by Hour')
ax.set_xlabel('hour_of_day')

plt.tight_layout()
plt.show()
print("""
📋 Evaluation — hour_of_day
   VERY USEFUL: Strong non-linear effect. Demand peaks at 7-9am and 4-7pm.
   Ratio dips sharply during peak hours. Critical temporal feature.
   Encode as cyclic (sin/cos) for gradient-based models; tree models handle raw integers.
""")

In [ ]:
# ─── 4.23 day_of_week ─────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
day_labels = ['Sun', 'Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat']
daily_ratio = pdf_all.groupby('day_of_week')['supply_demand_ratio'].mean()
ax.bar(daily_ratio.index, daily_ratio.values, color=PALETTE[4])
ax.set_xticks(range(1, 8))
ax.set_xticklabels(day_labels)
ax.set_title('Average supply_demand_ratio by Day of Week')
ax.set_ylabel('supply_demand_ratio')
plt.tight_layout()
plt.show()
print("""
📋 Evaluation — day_of_week
   USEFUL: Weekday vs weekend show distinct demand patterns.
   Is_weekend captures most of this, but day_of_week provides finer granularity
   (Friday nights vs Monday mornings differ significantly).
   Keep both for tree-based models.
""")

In [ ]:
# ─── 4.24 is_weekend ──────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 4))
pdf_all.boxplot(column='supply_demand_ratio', by='is_weekend', ax=ax)
ax.set_title('Ratio: Weekday vs Weekend')
ax.set_xlabel('is_weekend (0=weekday, 1=weekend)')
ax.set_ylabel('supply_demand_ratio')
plt.tight_layout()
plt.show()
print("""
📋 Evaluation — is_weekend
   USEFUL: Binary indicator capturing weekend surge and night-life demand.
   Partially redundant with day_of_week but very clean and useful.
   Keep as explicit feature.
""")

In [ ]:
# ─── 4.25 neighbor_requests_30m ───────────────────────────────────────────────
validate_feature(
    'neighbor_requests_30m', pdf_all, pdf_zone,
    """USEFUL: Captures demand spillover from adjacent zones.
    If neighboring zones are saturated, demand may spill into this zone.
    Adds genuinely new spatial information not captured by zone-level features.
    Quality depends on adjacency map accuracy."""
)

---
## Section 5 — Lag Analysis

We test lag values from 1 to 12 (5-minute increments, covering 60 minutes) and measure their Pearson correlation with `supply_demand_ratio`.

In [ ]:
# ─── 5.1 Compute Lags 1–12 in Spark ──────────────────────────────────────────
MAX_LAG = 12

gold_lags = gold_labeled
for lag_n in range(1, MAX_LAG + 1):
    gold_lags = gold_lags.withColumn(
        f'lag_{lag_n}',
        F.lag('requests_30m', lag_n).over(zone_time_window)
    )

# Sample to Pandas for correlation analysis
lag_cols = [f'lag_{n}' for n in range(1, MAX_LAG + 1)]
pdf_lags = (
    gold_lags
    .select(['supply_demand_ratio'] + lag_cols)
    .dropna()
    .sample(fraction=0.1, seed=42)
    .limit(20_000)
    .toPandas()
)

print(f'Lag sample shape: {pdf_lags.shape}')

In [ ]:
# ─── 5.2 Correlation vs Lag Plot ──────────────────────────────────────────────
lag_corrs = [
    pdf_lags[f'lag_{n}'].corr(pdf_lags['supply_demand_ratio'])
    for n in range(1, MAX_LAG + 1)
]
lag_minutes = [n * 5 for n in range(1, MAX_LAG + 1)]

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(lag_minutes, lag_corrs, marker='o', color=PALETTE[0], linewidth=2)
ax.axhline(0, color='gray', linestyle='--', linewidth=0.8)
ax.fill_between(lag_minutes, lag_corrs, alpha=0.15, color=PALETTE[0])
ax.set_xlabel('Lag (minutes)')
ax.set_ylabel('Pearson r with supply_demand_ratio')
ax.set_title('Autocorrelation of requests_30m vs Supply-Demand Ratio at Different Lags')
ax.set_xticks(lag_minutes)
ax.xaxis.set_major_formatter(mticker.FormatStrFormatter('%d min'))
for i, (x, y) in enumerate(zip(lag_minutes, lag_corrs)):
    ax.annotate(f'{y:.3f}', (x, y), textcoords='offset points', xytext=(0, 8),
                ha='center', fontsize=8)
plt.tight_layout()
plt.show()

best_lag = lag_minutes[int(np.argmax(np.abs(lag_corrs)))]
print(f'\n✅ Best lag: {best_lag} minutes (lag_{best_lag//5})')
print('\nRecommendation:')
print(' - Lag 5 min  (lag_1): captures immediate momentum')
print(' - Lag 30 min (lag_6): captures same-window one period ago')
print(' - Add lag_12 (60 min ago) if correlation remains above 0.10')

---
## Section 6 — Exploratory Data Analysis (EDA)

In [ ]:
# ─── 6.1 Demand vs Supply Over Time (Zone 161) ────────────────────────────────
fig, ax = plt.subplots(figsize=(16, 5))
ax.plot(pdf_zone['window_end'], pdf_zone['requests_30m'],
        label='Demand (requests)', color=PALETTE[0], linewidth=0.9)
ax.plot(pdf_zone['window_end'], pdf_zone['completed_30m'],
        label='Supply (completions)', color=PALETTE[1], linewidth=0.9)
ax.fill_between(
    pdf_zone['window_end'],
    pdf_zone['requests_30m'],
    pdf_zone['completed_30m'],
    where=(pdf_zone['requests_30m'] > pdf_zone['completed_30m']),
    alpha=0.25, color='red', label='Demand > Supply'
)
ax.fill_between(
    pdf_zone['window_end'],
    pdf_zone['requests_30m'],
    pdf_zone['completed_30m'],
    where=(pdf_zone['requests_30m'] <= pdf_zone['completed_30m']),
    alpha=0.2, color='green', label='Supply >= Demand'
)
ax.set_title(f'Demand vs Supply Over Time — Zone {SAMPLE_ZONE}')
ax.set_xlabel('Time')
ax.set_ylabel('Trips per 30-min window')
ax.legend()
ax.tick_params(axis='x', rotation=20)
plt.tight_layout()
plt.show()

In [ ]:
# ─── 6.2 Distribution of supply_demand_ratio ──────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

ratio_clipped = pdf_all['supply_demand_ratio'].clip(0, 3)

# Histogram
axes[0].hist(ratio_clipped, bins=80, color=PALETTE[0], edgecolor='white', linewidth=0.3)
axes[0].axvline(1.0, color='red', linestyle='--', label='ratio = 1.0')
axes[0].axvline(Q1, color='orange', linestyle=':', label=f'Q1={Q1:.2f}')
axes[0].axvline(Q2, color='green',  linestyle=':', label=f'Q2={Q2:.2f}')
axes[0].axvline(Q3, color='blue',   linestyle=':', label=f'Q3={Q3:.2f}')
axes[0].set_title('Distribution of supply_demand_ratio')
axes[0].set_xlabel('supply_demand_ratio')
axes[0].legend(fontsize=8)

# Log-scale
axes[1].hist(ratio_clipped, bins=80, color=PALETTE[1], edgecolor='white', linewidth=0.3)
axes[1].set_yscale('log')
axes[1].axvline(1.0, color='red', linestyle='--')
axes[1].set_title('Distribution (log y-scale)')
axes[1].set_xlabel('supply_demand_ratio')

# KDE
ratio_clipped.plot.kde(ax=axes[2], color=PALETTE[2])
axes[2].axvline(1.0, color='red', linestyle='--', label='ratio=1')
axes[2].set_title('KDE of supply_demand_ratio')
axes[2].set_xlabel('supply_demand_ratio')
axes[2].legend()

plt.tight_layout()
plt.show()

print(pdf_all['supply_demand_ratio'].describe())
print(f'\n% under-supplied (ratio < 1): {(pdf_all["supply_demand_ratio"] < 1).mean()*100:.1f}%')

In [ ]:
# ─── 6.3 Behavior During Imbalance ───────────────────────────────────────────
# Compare feature distributions in under-supplied vs surplus windows

under   = pdf_all[pdf_all['supply_demand_ratio'] < 1.0]
surplus = pdf_all[pdf_all['supply_demand_ratio'] >= 1.0]

compare_feats = ['pickup_delay_mean', 'avg_fare', 'avg_speed', 'requests_30m', 'inflow_30m']

fig, axes = plt.subplots(1, len(compare_feats), figsize=(20, 4))
for ax, feat in zip(axes, compare_feats):
    ax.hist(under[feat].dropna().clip(*np.nanpercentile(under[feat].dropna(), [1, 99])),
            bins=40, alpha=0.6, color='red', label='Under-supplied', density=True)
    ax.hist(surplus[feat].dropna().clip(*np.nanpercentile(surplus[feat].dropna(), [1, 99])),
            bins=40, alpha=0.6, color='green', label='Surplus', density=True)
    ax.set_title(feat)
    ax.legend(fontsize=7)

fig.suptitle('Feature Distributions: Under-supplied vs Surplus Windows', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ─── 6.4 Correlation Heatmap ──────────────────────────────────────────────────
corr_feats = [
    'requests_30m', 'lag_1_requests', 'lag_6_requests',
    'completed_30m', 'matched_30m', 'pickup_delay_mean',
    'inflow_30m', 'outflow_30m', 'avg_fare', 'avg_speed',
    'avg_distance', 'neighbor_requests_30m', 'supply_demand_ratio'
]

corr_matrix = pdf_all[corr_feats].corr()

fig, ax = plt.subplots(figsize=(14, 11))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt='.2f',
    cmap='RdYlGn', center=0, ax=ax,
    linewidths=0.5, annot_kws={'size': 8}
)
ax.set_title('Feature Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ─── 6.5 Hourly x Day-of-Week Demand Heatmap ─────────────────────────────────
pivot = pdf_all.pivot_table(
    values='requests_30m',
    index='hour_of_day',
    columns='day_of_week',
    aggfunc='mean'
)
pivot.columns = ['Sun','Mon','Tue','Wed','Thu','Fri','Sat']

fig, ax = plt.subplots(figsize=(12, 7))
sns.heatmap(pivot, cmap='YlOrRd', ax=ax, fmt='.0f', annot=True, annot_kws={'size': 8})
ax.set_title('Average Requests per 30-min Window: Hour × Day', fontsize=13, fontweight='bold')
ax.set_xlabel('Day of Week')
ax.set_ylabel('Hour of Day')
plt.tight_layout()
plt.show()

---
## Section 7 — Feature Importance

We train a LightGBM regressor on `supply_demand_ratio` (continuous target) to get global feature importance. This model is **not the final production model** — it's a diagnostic tool.

In [ ]:
# ─── 7.1 Prepare Training Data ────────────────────────────────────────────────
FEATURE_COLS = [
    'requests_30m', 'requests_5m', 'lag_1_requests', 'lag_6_requests',
    'completed_30m', 'completed_5m',
    'matched_30m', 'matched_5m', 'pickup_delay_mean', 'pickup_delay_std',
    'inflow_30m', 'outflow_30m',
    'avg_speed', 'std_speed', 'avg_trip_time', 'std_trip_time',
    'avg_fare', 'std_fare', 'avg_driver_pay',
    'avg_distance', 'std_distance',
    'hour_of_day', 'day_of_week', 'is_weekend',
    'neighbor_requests_30m',
]

TARGET = 'supply_demand_ratio'

df_model = pdf_all[FEATURE_COLS + [TARGET]].dropna()
# Clip extreme ratio values
df_model[TARGET] = df_model[TARGET].clip(0, 5)

X = df_model[FEATURE_COLS]
y = df_model[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Train: {X_train.shape}  Test: {X_test.shape}')

In [ ]:
# ─── 7.2 Train LightGBM Regressor ─────────────────────────────────────────────
lgb_params = {
    'objective':        'regression_l1',
    'n_estimators':     500,
    'learning_rate':    0.05,
    'num_leaves':       63,
    'max_depth':        -1,
    'min_child_samples': 20,
    'subsample':        0.8,
    'colsample_bytree': 0.8,
    'reg_alpha':        0.1,
    'reg_lambda':       0.1,
    'random_state':     42,
    'n_jobs':           -1,
    'verbose':          -1,
}

reg_model = lgb.LGBMRegressor(**lgb_params)
reg_model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(100)]
)

y_pred = reg_model.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2   = r2_score(y_test, y_pred)
print(f'\nBaseline LightGBM Regressor:')
print(f'  RMSE = {rmse:.4f}')
print(f'  R²   = {r2:.4f}')

In [ ]:
# ─── 7.3 Feature Importance Plot ──────────────────────────────────────────────
importance_df = pd.DataFrame({
    'feature':   FEATURE_COLS,
    'gain':      reg_model.booster_.feature_importance(importance_type='gain'),
    'split':     reg_model.booster_.feature_importance(importance_type='split'),
}).sort_values('gain', ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 8))

for ax, col, title in zip(axes, ['gain', 'split'], ['Gain Importance', 'Split Importance']):
    ax.barh(importance_df['feature'], importance_df[col], color=PALETTE[0])
    ax.set_title(title)
    ax.set_xlabel('Importance')
    ax.set_ylabel('Feature')

plt.suptitle('LightGBM Feature Importance — Regressor', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('\nTop 10 features by Gain:')
print(importance_df.sort_values('gain', ascending=False).head(10)[['feature', 'gain']].to_string(index=False))

---
## Section 8 — Label Evaluation

We train a LightGBM classifier for each labeling strategy and compare:
- Class distribution
- AUC / F1 / Weighted Accuracy

In [ ]:
# ─── 8.1 Class Distribution ───────────────────────────────────────────────────
label_strategies = {
    'binary':  'label_binary',
    '3-class': 'label_3class',
    '4-class': 'label_4class',
}

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (name, col) in zip(axes, label_strategies.items()):
    if col in pdf_all.columns:
        counts = pdf_all[col].value_counts().sort_index()
        ax.bar(counts.index.astype(str), counts.values, color=PALETTE[:len(counts)])
        ax.set_title(f'{name} — Class Distribution')
        ax.set_xlabel('Class')
        ax.set_ylabel('Count')
        for i, (cls, cnt) in enumerate(counts.items()):
            ax.text(i, cnt, f'{cnt/len(pdf_all)*100:.1f}%', ha='center', va='bottom', fontsize=9)

plt.suptitle('Label Distribution by Strategy', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ─── 8.2 Train & Compare Classifiers ──────────────────────────────────────────

clf_params = {
    'n_estimators':      400,
    'learning_rate':     0.05,
    'num_leaves':        63,
    'max_depth':         -1,
    'subsample':         0.8,
    'colsample_bytree':  0.8,
    'random_state':      42,
    'n_jobs':            -1,
    'verbose':           -1,
}

results = {}

for name, label_col in label_strategies.items():
    if label_col not in pdf_all.columns:
        print(f'Skipping {name} — column not found in sample')
        continue

    df_clf = pdf_all[FEATURE_COLS + [label_col]].dropna()
    X_c = df_clf[FEATURE_COLS]
    y_c = df_clf[label_col]

    X_tr, X_te, y_tr, y_te = train_test_split(
        X_c, y_c, test_size=0.2, random_state=42, stratify=y_c
    )

    n_classes = y_c.nunique()
    obj = 'binary' if n_classes == 2 else 'multiclass'

    clf = lgb.LGBMClassifier(objective=obj, num_class=(n_classes if n_classes > 2 else None),
                              **clf_params)
    clf.fit(X_tr, y_tr,
            eval_set=[(X_te, y_te)],
            callbacks=[lgb.early_stopping(40, verbose=False), lgb.log_evaluation(-1)])

    y_pred_clf = clf.predict(X_te)
    y_prob_clf = clf.predict_proba(X_te)

    # AUC
    if n_classes == 2:
        auc = roc_auc_score(y_te, y_prob_clf[:, 1])
    else:
        auc = roc_auc_score(
            label_binarize(y_te, classes=sorted(y_c.unique())),
            y_prob_clf, multi_class='ovr', average='weighted'
        )

    report = classification_report(y_te, y_pred_clf, output_dict=True)
    results[name] = {
        'model':    clf,
        'auc':      auc,
        'f1':       report['weighted avg']['f1-score'],
        'accuracy': report['accuracy'],
        'y_te':     y_te,
        'y_pred':   y_pred_clf,
        'n_classes': n_classes,
    }
    print(f'[{name}] AUC={auc:.4f}  F1={report["weighted avg"]["f1-score"]:.4f}  Acc={report["accuracy"]:.4f}')

In [ ]:
# ─── 8.3 Confusion Matrices + Performance Summary ─────────────────────────────
fig, axes = plt.subplots(1, len(results), figsize=(7 * len(results), 6))
if len(results) == 1:
    axes = [axes]

for ax, (name, res) in zip(axes, results.items()):
    cm = confusion_matrix(res['y_te'], res['y_pred'])
    disp = ConfusionMatrixDisplay(confusion_matrix=cm)
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(f'{name}\nAUC={res["auc"]:.3f}  F1={res["f1"]:.3f}')

plt.suptitle('Confusion Matrices by Label Strategy', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Summary table
summary = pd.DataFrame({
    name: {'AUC': r['auc'], 'Weighted F1': r['f1'], 'Accuracy': r['accuracy']}
    for name, r in results.items()
}).T
print('\n── Performance Summary ──')
print(summary.to_string())

In [ ]:
# ─── 8.4 Detailed Classification Report for Best Strategy ─────────────────────
best_strategy = max(results, key=lambda k: results[k]['f1'])
best = results[best_strategy]

print(f'\n🏆 Best Strategy: {best_strategy}  (F1={best["f1"]:.4f}, AUC={best["auc"]:.4f})')
print('\nDetailed Classification Report:')
print(classification_report(best['y_te'], best['y_pred']))

---
## Section 9 — Final Recommendations

Based on the full analysis above, this section documents the recommended production configuration.

In [ ]:
# ─── 9.1 Final Feature Set ────────────────────────────────────────────────────

FINAL_FEATURES = {
    'Demand (keep)': [
        'requests_30m',      # Core demand signal
        'requests_5m',       # Momentum
        'lag_1_requests',    # 5-min autocorrelation
        'lag_6_requests',    # 30-min autocorrelation
    ],
    'Supply (keep)': [
        'completed_30m',     # Core supply signal
        # completed_5m → DROP if VIF > 5 with completed_30m
    ],
    'Matching (keep)': [
        'matched_30m',
        'pickup_delay_mean', # Strongest single feature for imbalance
        'pickup_delay_std',
        # matched_5m → DROP if redundant
    ],
    'Flow (keep)': [
        'inflow_30m',
        'outflow_30m',
    ],
    'Traffic (keep)': [
        'avg_speed',         # Congestion proxy
        'avg_trip_time',     # Driver lock-out proxy
        # Drop: std_speed, std_trip_time
    ],
    'Price (keep)': [
        'avg_fare',          # Surge proxy
        'avg_driver_pay',
        # Drop: std_fare
    ],
    'Trip (keep)': [
        'avg_distance',
        # Drop: std_distance
    ],
    'Temporal (keep)': [
        'hour_of_day',
        'day_of_week',
        'is_weekend',
    ],
    'Spatial (keep)': [
        'neighbor_requests_30m',
    ],
    'DROPPED (redundant/marginal)': [
        'completed_5m', 'matched_5m',
        'std_speed', 'std_trip_time', 'std_fare', 'std_distance',
    ]
}

print('=== FINAL FEATURE SET ===')
for group, feats in FINAL_FEATURES.items():
    print(f'\n{group}:')
    for f in feats:
        print(f'  - {f}')

In [ ]:
# ─── 9.2 Final Lag Selection ──────────────────────────────────────────────────

print('=== FINAL LAG SELECTION ===')
print('''
Recommended lags for requests_30m:
  ✅ lag_1  (5 min ago)  — captures immediate demand momentum
  ✅ lag_6  (30 min ago) — captures same-period trend
  ⚠️  lag_12 (60 min ago) — add only if |r| > 0.10 in lag analysis

For production, also consider:
  - lag_288 (24 hours ago) — same time yesterday
  - lag_2016 (7 days ago)  — same time last week
  → These require a full year of history and joining on (zone_id, hour, day_of_week).
''')

In [ ]:
# ─── 9.3 Final Label Strategy ─────────────────────────────────────────────────

print('=== FINAL LABEL RECOMMENDATION ===')
print(f'''
Based on AUC / F1 comparison:

  🏆 RECOMMENDED: 3-class labeling

  Rationale:
    - Binary is too coarse: "under-supplied" masks severity gradient.
      A zone at ratio=0.9 and ratio=0.1 need very different interventions.
    - 4-class produces unbalanced classes (Q1/Q3 quantile cuts
      may be too tight), hurting minority-class F1.
    - 3-class (severe shortage / mild shortage / surplus) provides
      actionable granularity while maintaining balanced classes.

  Thresholds (data-driven from quantiles):
    Class 0 (severe shortage):  ratio < {Q1:.3f}
    Class 1 (mild shortage):    {Q1:.3f} ≤ ratio < 1.0
    Class 2 (surplus):          ratio ≥ 1.0

  Operational mapping:
    Class 0 → Immediate driver reallocation required
    Class 1 → Monitor, pre-position nearby drivers
    Class 2 → No action needed

  Alternative: If the business needs a regression output for ranking
  zones, use supply_demand_ratio directly with the LightGBM regressor
  (RMSE-L1 objective, clip ratio at [0, 3]).
''')

In [ ]:
# ─── 9.4 Production Pipeline Summary ─────────────────────────────────────────

print('''
=== PRODUCTION PIPELINE SUMMARY ===

Bronze → Silver → Gold:
  1. Bronze: raw HVFHV Parquet (year/month partitioned)
  2. Silver: cleaned trips with derived columns (speed, delay, etc.)
  3. Gold: aggregated (zone_id, window_end) with all features + labels

Gold table schema (final):
  Primary key : (zone_id INT, window_end TIMESTAMP)
  Features    : 20 columns (see final feature set above)
  Labels      : supply_demand_ratio DOUBLE, label_3class INT
  Partitioned by: year, month, zone_id

Training cadence:
  - Re-train weekly on rolling 90-day window
  - Evaluate on last 7 days (held-out)

Serving:
  - Real-time scoring: every 5 minutes per zone
  - Latency budget: <1s per zone batch
  - Model export: LightGBM .txt or MLflow artifact

Monitoring:
  - Track distribution shift on requests_30m, pickup_delay_mean
  - Alert if class 0 rate deviates > 2σ from rolling baseline
''')

In [ ]:
# ─── Cleanup ──────────────────────────────────────────────────────────────────
spark.stop()
print('Spark session stopped. Notebook complete. ✅')